<a href="https://colab.research.google.com/github/mithun30052001/capstone-project/blob/main/Part4-LLM-Feature/LLM_Intelligent_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import userdata

# Retrieve the key securely from Colab Secrets
try:
    os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
    print("API Key successfully loaded from Colab Secrets!")
except Exception as e:
    print("Error: Make sure you added 'LLM_API_KEY' in the Colab Secrets sidebar (Key icon 🔑) and enabled Notebook Access.")

API Key successfully loaded from Colab Secrets!


In [5]:
import os
from google.colab import userdata

try:
    key = userdata.get('LLM_API_KEY')
    if key is None or key == "":
        print("❌ Diagnostic: Secret exists but the value is empty!")
    else:
        print(f"✅ Diagnostic: Secret successfully read! (Starts with: {key[:8]}...)")
        # Load it into the environment variables
        os.environ['LLM_API_KEY'] = key
except Exception as e:
    print(f"❌ Diagnostic: Failed to read secret. Error: {str(e)}")
    print("👉 ACTION REQUIRED: Go to the left sidebar (Key icon 🔑), find 'LLM_API_KEY', and make sure the blue 'Notebook access' toggle is switched ON.")

✅ Diagnostic: Secret successfully read! (Starts with: sk-or-v1...)


In [7]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    # Retrieve the API key directly
    api_key = os.environ.get('LLM_API_KEY')
    if not api_key:
        print("❌ Error: LLM_API_KEY environment variable is empty. Did you run the Step 1 diagnostic cell?")
        return "ERROR: API KEY MISSING"

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        # OpenRouter sometimes requests these headers for ranking metrics
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "IIT Capstone Project"
    }
    payload = {
        "model": "google/gemini-2.5-flash",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ API Error {response.status_code}: {response.text}")
            return f"ERROR: API returned status {response.status_code}"
    except Exception as e:
        print(f"❌ Request failed: {e}")
        return "ERROR: Connection failed"

In [9]:
# Test Call
print("Testing LLM API Connection...")
test_resp = call_llm("You are a simple chatbot.", "Reply with only the word: hello", temperature=0.0)
print(f"LLM Response: '{test_resp.strip()}' (Should say 'hello')")

Testing LLM API Connection...
LLM Response: 'hello' (Should say 'hello')


In [11]:
# Define JSON validation schema (Requires at least 5 scalar fields)
EXPLANATION_SCHEMA = {
    "type": "object",
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"],
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    }
}

SYSTEM_PROMPT = """
You are a highly analytical AI assistant specializing in explainable machine learning models for FinTech systems.
Your task is to explain a model's classification prediction given its input features, the predicted class, and the prediction probability.

RULES:
1. Output ONLY a valid JSON object. No explanations, no markdown formatting (do not wrap in ```json), and no surrounding conversational text.
2. The JSON output must strictly conform to the target schema.
3. Be professional and objective in your reasons.
4. Top reasons must directly link feature values to the prediction.
"""

USER_PROMPT_TEMPLATE = """
Explain the following machine learning model prediction:

--- INPUT DATA ---
Features: {features}
Predicted Class: {predicted_class} (where 1 = Extreme Delay, 0 = Normal Speed)
Prediction Probability: {probability:.4f}

--- SCHEMA REQUIREMENT ---
Return a JSON object containing exactly these fields:
{{
    "prediction_label": "string: 'Extreme Delay' or 'Normal Speed'",
    "confidence_level": "string: 'low' | 'medium' | 'high'",
    "top_reason": "string: primary feature driving this prediction",
    "second_reason": "string: secondary feature supporting this prediction",
    "next_step": "string: recommended operational next step based on the prediction"
}}
"""

In [13]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# Test Guardrails
clean_input = "Consumer complaint regarding a credit card charge of $50.00."
pii_input = "My email is user@test.com. Please refund my invoice of $50.00."

print(f"Clean input: '{clean_input}' -> PII Detected: {has_pii(clean_input)} (Should be False)")
print(f"PII input: '{pii_input}' -> PII Detected: {has_pii(pii_input)} (Should be True)")

Clean input: 'Consumer complaint regarding a credit card charge of $50.00.' -> PII Detected: False (Should be False)
PII input: 'My email is user@test.com. Please refund my invoice of $50.00.' -> PII Detected: True (Should be True)


In [16]:
# Load Model from Part 3
print("Loading model...")
model = joblib.load('best_model.pkl')

# FIX: Get feature names directly from the Pipeline object, not the classifier step
feature_cols = model.feature_names_in_

# Helper function to align user features to model structure
def encode_record(features_dict):
    row = {col: 0 for col in feature_cols}
    row['narrative_length'] = features_dict.get('narrative_length', 100)
    row['company_response_length'] = features_dict.get('company_response_length', 50)
    row['ZIP code numeric'] = features_dict.get('zip_code', 0)
    row['Submitted via_encoded'] = features_dict.get('submitted_via', 1)

    product = features_dict.get('product', '')
    prod_col = f"Product_{product}"
    if prod_col in row:
        row[prod_col] = 1

    return pd.DataFrame([row])[feature_cols]

print("Model loaded and helper function defined successfully!")

Loading model...
Model loaded and helper function defined successfully!


In [18]:
# Run A/B test comparison
sample_features = {'product': 'Mortgage', 'submitted_via': 3, 'narrative_length': 2500}
encoded_sample = encode_record(sample_features)

pred = int(model.predict(encoded_sample)[0])
prob = float(model.predict_proba(encoded_sample)[0][pred])

formatted_user_prompt = USER_PROMPT_TEMPLATE.format(
    features=sample_features, predicted_class=pred, probability=prob
)

print("--- Calling LLM with Temperature = 0.0 ---")
resp_temp0 = call_llm(SYSTEM_PROMPT, formatted_user_prompt, temperature=0.0)
print(resp_temp0)

print("\n--- Calling LLM with Temperature = 0.7 ---")
resp_temp07 = call_llm(SYSTEM_PROMPT, formatted_user_prompt, temperature=0.7)
print(resp_temp07)

--- Calling LLM with Temperature = 0.0 ---
{"prediction_label": "Normal Speed", "confidence_level": "high", "top_reason": "The 'product' type 'Mortgage' is strongly associated with a 'Normal Speed' classification.", "second_reason": "The 'submitted_via' value of 3 is a contributing factor to the 'Normal Speed' prediction.", "next_step": "No immediate intervention is required as the submission is predicted to proceed at a normal speed."}

--- Calling LLM with Temperature = 0.7 ---
{"prediction_label": "Normal Speed", "confidence_level": "high", "top_reason": "The 'product' type 'Mortgage' is strongly associated with a Normal Speed classification.", "second_reason": "The 'submitted_via' value of 3 indicates a submission channel typically resulting in Normal Speed processing.", "next_step": "Monitor the processing of this mortgage application to ensure it remains within normal operational parameters."}


In [19]:
# 3 Hand-crafted inputs
test_inputs = [
    # Input 1: Web submission, short text (Low risk of delay)
    {
        "product": "Credit card or prepaid card", "submitted_via": 1,
        "zip_code": 90210, "narrative_length": 250, "company_response_length": 80,
        "narrative": "Quick billing dispute regarding an overcharge."
    },
    # Input 2: Paper mail submission, long text (High risk of delay)
    {
        "product": "Mortgage", "submitted_via": 3,
        "zip_code": 10001, "narrative_length": 3500, "company_response_length": 450,
        "narrative": "I have contacted customer service multiple times regarding my mortgage payment processing delays..."
    },
    # Input 3: Submission containing PII (Should trigger guardrail)
    {
        "product": "Debt collection", "submitted_via": 2,
        "zip_code": 30301, "narrative_length": 500, "company_response_length": 150,
        "narrative": "My phone is 123-456-7890. Please call me regarding this collection notice."
    }
]

pipeline_results = []

for idx, record in enumerate(test_inputs):
    print(f"\nProcessing Input {idx + 1}...")

    # 1. Check Guardrails first
    if has_pii(record['narrative']):
        print("Input blocked: PII detected.")
        pipeline_results.append({
            "Input": record, "Class": "N/A", "Prob": "N/A",
            "Output": "BLOCKED (PII)", "Status": "Blocked"
        })
        continue

    # 2. Encode and Predict
    encoded_x = encode_record(record)
    pred_class = int(model.predict(encoded_x)[0])
    pred_prob = float(model.predict_proba(encoded_x)[0][pred_class])

    # 3. Format Prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(
        features=record, predicted_class=pred_class, probability=pred_prob
    )

    # 4. Call LLM
    raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)

    # 5. Clean, Parse and Validate Schema
    validation = "Pass"
    explanation_dict = None
    if raw_response:
        try:
            cleaned = raw_response.strip()
            if cleaned.startswith("```json"):
                cleaned = cleaned[7:-3].strip()
            elif cleaned.startswith("```"):
                cleaned = cleaned[3:-3].strip()

            explanation_dict = json.loads(cleaned)
            validate(instance=explanation_dict, schema=EXPLANATION_SCHEMA)
        except (json.JSONDecodeError, ValidationError) as e:
            validation = f"Fail: {str(e)}"
            # Fallback
            explanation_dict = {
                "prediction_label": "Extreme Delay" if pred_class == 1 else "Normal Speed",
                "confidence_level": "medium",
                "top_reason": "Fallback: High processing features.",
                "second_reason": f"Probability: {pred_prob:.2%}",
                "next_step": "Manual triage required."
            }
    else:
        validation = "API Error"

    print(f"ML Prediction: {pred_class} | Probability: {pred_prob:.4f}")
    print(f"Raw LLM Response: {raw_response}")
    print(f"Validation Status: {validation}")

    pipeline_results.append({
        "Input": record, "Class": pred_class, "Prob": f"{pred_prob:.4f}",
        "Output": explanation_dict, "Status": validation
    })

print("\n--- Pipeline Run Summary ---")
display(pd.DataFrame(pipeline_results))


Processing Input 1...
ML Prediction: 0 | Probability: 0.9510
Raw LLM Response: {"prediction_label": "Normal Speed", "confidence_level": "high", "top_reason": "The narrative length is 250 characters, which is within the typical range for normal speed resolutions.", "second_reason": "The company response length is 80 characters, indicating a concise and likely efficient resolution process.", "next_step": "Process the complaint as a standard case, anticipating a normal resolution timeframe."}
Validation Status: Pass

Processing Input 2...
ML Prediction: 0 | Probability: 0.9406
Raw LLM Response: {"prediction_label": "Normal Speed", "confidence_level": "high", "top_reason": "The company response length is 450, which is indicative of a comprehensive response and typically associated with normal processing speeds.", "second_reason": "The narrative length is 3500, suggesting a detailed complaint, but this is offset by the comprehensive company response.", "next_step": "Monitor the case for an

,Input,Class,Prob,Output,Status
0,"{'product': 'Credit card or prepaid card', 'su...",0,0.9510,"{'prediction_label': 'Normal Speed', 'confiden...",Pass
1,"{'product': 'Mortgage', 'submitted_via': 3, 'z...",0,0.9406,"{'prediction_label': 'Normal Speed', 'confiden...",Pass
2,"{'product': 'Debt collection', 'submitted_via'...",N/A,N/A,BLOCKED (PII),Blocked
